# 连续 Label 主线 Layer Probe

主目标是 `residual_signed_raw` 与 `delta_log_dispersion` 在 Layer 0～12 的信息演化、关联和可解码性。固定头路径按当前协议锁定为 `CLS → fc`，属于用户锁定假设，不代表已考证原训练架构。旧情绪主线产物不在本 Notebook 中读取。

In [ ]:
from pathlib import Path
import json
import os
import pandas as pd
from IPython.display import display

CONFIGURED_ROOT = os.environ.get('CHINESE_WWM_ROBERTA_ROOT')
ROOT_CANDIDATES = [
    Path(CONFIGURED_ROOT).expanduser() if CONFIGURED_ROOT else None,
    Path.cwd().resolve(),
    Path.cwd().resolve().parent,
    Path.home() / 'Projects' / 'chinese-wwm-roberta',
]
ROOT = next((path.resolve() for path in ROOT_CANDIDATES if path is not None and (path / 'configs' / 'layer_probe.yaml').is_file()), None)
assert ROOT is not None, '找不到项目根目录；请设置CHINESE_WWM_ROBERTA_ROOT或确认~/Projects/chinese-wwm-roberta存在'
os.chdir(ROOT)

from src.layer_probe_representations import plot_representation_layer_norms, run_representation_stage, resolve_representation_directory, validate_representation_artifacts
from src.layer_probe_continuous import (align_continuous_targets, plot_aligned_target_counts, plot_continuous_probe_curves, plot_fixed_head_convergence, plot_fixed_head_label_associations, run_continuous_probe_stage, run_fixed_head_analysis_stage, run_fixed_head_label_stage, validate_aligned_targets, validate_continuous_probe_outputs, validate_fixed_head_analysis_outputs, validate_fixed_head_label_outputs)
from src.layer_probe_models import plot_optional_sentiment_curve, run_optional_sentiment_probe_stage, run_return_probe_stage, validate_optional_sentiment_outputs
from src.layer_probe_panel import plot_stock_day_summary, run_stock_day_panel_stage, validate_stock_day_artifacts
from src.layer_probe_factors import run_factor_validation_stage
from src.layer_probe_pipeline import (assert_preflight, load_layer_probe_config, plot_factor_summary, plot_return_curve, preflight_report, run_final_test_once, validate_factor_outputs, validate_pipeline_outputs, validate_return_probe_outputs)

CONFIG_PATH = ROOT / 'configs' / 'layer_probe.yaml'
config = load_layer_probe_config(CONFIG_PATH)
RUN_DIR = Path(config['output']['run_directory']).expanduser().resolve()
pd.set_option('display.max_columns', 100)
print('repo:', ROOT)
print('run:', RUN_DIR)

## 0. 执行开关

已有且通过manifest验收的阶段应把对应开关设为 `False`。最终test与情绪附录默认关闭。

In [ ]:
RUN_STAGE_1_REPRESENTATIONS = True
RUN_STAGE_2A_FIXED_HEAD = True
RUN_STAGE_2A_PLUS_ASSOCIATION = True
RUN_STAGE_2B_CONTINUOUS_PROBE = True
RUN_STAGE_3_STOCK_DAY = True
RUN_STAGE_4_RETURN_PROBE = True
RUN_STAGE_5_6_FACTOR_VALIDATION = True
RUN_FINAL_TEST_CELL = False
RUN_SENTIMENT_APPENDIX = False

if RUN_FINAL_TEST_CELL:
    assert config['strict_test']['open_final_test'] is True
else:
    assert config['strict_test']['open_final_test'] is False
if RUN_SENTIMENT_APPENDIX:
    assert config['sentiment_appendix']['enabled'] is True

## 1. 一次 backbone forward 提取 Layer 0～12

本阶段不检查任何Label、收益或时间切分。Layer 12投影必须与锁定的normal forward数值一致。

In [ ]:
representation_preflight = preflight_report(config, stages=['representation'])
display(representation_preflight)
assert_preflight(representation_preflight)
if RUN_STAGE_1_REPRESENTATIONS:
    run_representation_stage(config)
representation_directory = resolve_representation_directory(config)
representation_check = validate_representation_artifacts(representation_directory)
display(pd.Series(representation_check, name='representation_validation'))
representation_manifest = json.loads((representation_directory / 'representation_manifest.json').read_text(encoding='utf-8'))
display(pd.Series(representation_manifest['layer12_equivalence']))
plot_representation_layer_norms(representation_directory);
display(pd.Series(representation_manifest['head_contract']))
assert representation_manifest['layer12_equivalence']['passed'] is True

## 2A. 无Label固定CLS-fc逐层投影

这里只分析分布、饱和、相邻层变化、类别翻转以及向最终方向的收敛，不比较‘最佳层’。

In [ ]:
head_preflight = preflight_report(config, stages=['fixed_head'])
display(head_preflight)
assert_preflight(head_preflight)
if RUN_STAGE_2A_FIXED_HEAD:
    run_fixed_head_analysis_stage(config)
fixed_head_dir = RUN_DIR / 'fixed_head_analysis'
display(pd.Series(validate_fixed_head_analysis_outputs(fixed_head_dir)))
display(pd.read_csv(fixed_head_dir / 'fixed_head_layer_distributions.csv'))
display(pd.read_csv(fixed_head_dir / 'fixed_head_adjacent_transitions.csv'))
plot_fixed_head_convergence(fixed_head_dir);

## 2A+. 固定头输出与连续Label的关系

不训练参数。所有结果按task、层和split独立计算，validation分组边界只由train固定头输出确定。

In [ ]:
continuous_preflight = preflight_report(config, stages=['continuous_labels'])
display(continuous_preflight)
assert_preflight(continuous_preflight)
aligned_target_dir = align_continuous_targets(config)
display(pd.Series(validate_aligned_targets(aligned_target_dir)))
if RUN_STAGE_2A_PLUS_ASSOCIATION:
    run_fixed_head_label_stage(config, evaluation_split='validation')
association_dir = RUN_DIR / 'fixed_head_label' / 'validation'
display(pd.Series(validate_fixed_head_label_outputs(association_dir)))
association_metrics = pd.read_csv(association_dir / 'fixed_head_label_metrics.csv')
display(association_metrics[association_metrics['prediction_role'].eq('oos')].sort_values(['task_id', 'layer']))
plot_aligned_target_counts(aligned_target_dir);
plot_fixed_head_label_associations(association_dir);

## 2B. 连续Label逐层Ridge Probe

每个task和layer独立拟合weighted StandardScaler与Ridge；validation只由train拟合，test前按冻结alpha在train+validation重拟合。

In [ ]:
if RUN_STAGE_2B_CONTINUOUS_PROBE:
    run_continuous_probe_stage(config, evaluation_split='validation')
continuous_probe_dir = RUN_DIR / 'continuous_probe' / 'validation'
display(pd.Series(validate_continuous_probe_outputs(continuous_probe_dir)))
continuous_metrics = pd.read_csv(continuous_probe_dir / 'continuous_probe_metrics.csv')
display(continuous_metrics[continuous_metrics['prediction_role'].eq('oos')].sort_values(['task_id', 'layer']))
display(pd.read_csv(continuous_probe_dir / 'continuous_probe_eligibility.csv'))
plot_continuous_probe_curves(continuous_probe_dir, metric='spearman');

## 3. 股票日Representation与未来收益

按symbol × trading_date聚合13层，保存n_texts，连接未来1/5/20日行业调整收益并执行最长窗口purge。

In [ ]:
return_preflight = preflight_report(config, stages=['returns'])
display(return_preflight)
assert_preflight(return_preflight)
if RUN_STAGE_3_STOCK_DAY:
    run_stock_day_panel_stage(config, evaluation_split='validation')
stock_day_dir = RUN_DIR / 'stock_day_panel'
display(pd.Series(validate_stock_day_artifacts(stock_day_dir, evaluation_split='validation')))
stock_day_panel = pd.read_parquet(stock_day_dir / 'validation' / 'stock_day_panel.parquet')
display(stock_day_panel.groupby('split').agg(rows=('representation_row', 'size'), dates=('trading_date', 'nunique'), symbols=('symbol', 'nunique'), mean_texts=('n_texts', 'mean')))
plot_stock_day_summary(stock_day_dir, evaluation_split='validation');

## 4. 逐层收益Ridge Probe（validation）

In [ ]:
if RUN_STAGE_4_RETURN_PROBE:
    run_return_probe_stage(config, evaluation_split='validation')
return_probe_dir = RUN_DIR / 'return_probe' / 'validation'
display(pd.Series(validate_return_probe_outputs(return_probe_dir)))
display(pd.read_csv(return_probe_dir / 'return_probe_metrics.csv').sort_values(['split', 'layer']))
plot_return_curve(return_probe_dir, split='validation');

## 5～6. 跨层因子与严格validation检验

包含单层、共识、分歧、deep-minus-middle、deep residual和PCA；增量IC控制fixed_head_margin与log1p(n_texts)。

In [ ]:
if RUN_STAGE_5_6_FACTOR_VALIDATION:
    run_factor_validation_stage(config, evaluation_split='validation')
factor_dir = RUN_DIR / 'factor_validation' / 'validation'
display(pd.Series(validate_factor_outputs(factor_dir)))
factor_summary = pd.read_csv(factor_dir / 'factor_summary.csv').sort_values('rank_ic', ascending=False)
display(factor_summary)
display(pd.read_csv(factor_dir / 'quantile_monotonicity.csv'))
display(pd.read_csv(factor_dir / 'incremental_ic.csv'))
plot_factor_summary(factor_dir);

## Validation后预注册

在打开最终test前，将保留因子写入 `strict_test.selected_factors`，冻结validation manifest和决策规则。

In [ ]:
candidate_columns = pd.read_parquet(factor_dir / 'candidate_factor_matrix.parquet').columns
candidate_names = [name for name in candidate_columns if name.startswith('single_layer_') or name in {'layer_consensus', 'layer_disagreement', 'deep_minus_middle', 'deep_residual'} or name.startswith('layer_pca_')]
display(pd.Series(candidate_names, name='可预注册因子'))

## 最终test：只打开一次

只有在validation完成、因子预注册、配置中 `strict_test.open_final_test: true` 且下方开关为True时执行。marker创建后不得重跑或调参。

In [ ]:
if RUN_FINAL_TEST_CELL:
    final_config = load_layer_probe_config(CONFIG_PATH)
    strict_preflight = preflight_report(final_config, stages=['strict_test'])
    display(strict_preflight)
    assert_preflight(strict_preflight)
    display(run_final_test_once(final_config))
else:
    print('最终test保持关闭。')

## 附录：可选0/1情绪Probe

本附录默认关闭，只有显式启用且确认类别映射后才检查sentiment_label；其缺失不影响任何主阶段。

In [ ]:
sentiment_preflight = preflight_report(config, stages=['sentiment_appendix'])
display(sentiment_preflight)
if RUN_SENTIMENT_APPENDIX:
    assert config['sentiment_appendix']['enabled'] is True
    assert_preflight(sentiment_preflight)
    sentiment_dir = run_optional_sentiment_probe_stage(config)
    display(pd.Series(validate_optional_sentiment_outputs(sentiment_dir)))
    plot_optional_sentiment_curve(sentiment_dir);
else:
    print('情绪附录保持关闭。')

## 全管线验收

In [ ]:
pipeline_validation = validate_pipeline_outputs(config)
display(pipeline_validation)
assert not pipeline_validation['status'].eq('failed').any()